# Serving

This notebook configures a deployment for the Breast Cancer schema without writing any repository checkpoints. A temporary checkpoint is enough to demonstrate the serving surface and keep the example self-contained.

Serving starts with the same model schema used for training. The additional dependency is Pydantic, which defines the request object accepted by the deployment wrapper.

In [1]:
import os
import tempfile
from pathlib import Path

import pydantic
import polars as pl
import torch
from loguru import logger
from sklearn.datasets import load_breast_cancer

import json2vec as j2v
from json2vec.inference.deployment import Deployment

logger.remove()

The serving payload uses the same nested `measurements` shape as the fine-tuning tutorial. Keeping train and serve shapes aligned is the main reason schemas, queries, and preprocessors live together.

In [2]:
ROOT = "[*][*]"
cancer = load_breast_cancer()
measurement_names = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
feature_index = {name: list(cancer.feature_names).index(name) for name in measurement_names}
malignant = [index for index, target in enumerate(cancer.target) if target == 0][:16]
benign = [index for index, target in enumerate(cancer.target) if target == 1][:16]
indices = [*malignant, *benign]

records = pl.DataFrame(
    [
        {
            "measurements": [
                {"name": name.replace(" ", "_"), "value": float(cancer.data[index, column])}
                for name, column in feature_index.items()
            ],
            "diagnosis": cancer.target_names[int(cancer.target[index])],
        }
        for index in indices
    ],
    strict=False,
)

records.head()

sample_request = {"measurements": records.to_dicts()[0]["measurements"]}


class Request(pydantic.BaseModel):
    measurements: list[dict]

Request.model_validate(sample_request)

Request(measurements=[{'name': 'mean_radius', 'value': 17.99}, {'name': 'mean_texture', 'value': 10.38}, {'name': 'mean_perimeter', 'value': 122.8}, {'name': 'mean_area', 'value': 1001.0}, {'name': 'mean_smoothness', 'value': 0.1184}])

The model includes a `diagnosis` target so it can be trained or inspected like the fine-tuning example. The request model validates the raw API payload before JSON2Vec encodes it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("name", query=f"{ROOT}.measurements[*].name", max_vocab_size=16),
        j2v.Number("value", query=f"{ROOT}.measurements[*].value"),
        name="measurements",
        max_length=len(measurement_names),
    ),
    j2v.Category("diagnosis", query=f"{ROOT}.diagnosis", target=True, max_vocab_size=2),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)
_ = model.set(j2v.where("name") == "record", embed=True)

A temporary checkpoint exercises the real load path without committing artifacts. Loading the model back also proves that the serialized schema can rebuild the modules.

In [4]:
_checkpoint_dir = tempfile.TemporaryDirectory()
CHECKPOINT = Path(_checkpoint_dir.name) / "breast-cancer.ckpt"
model.save(CHECKPOINT)

inspection = j2v.Model.load(CHECKPOINT)
_ = inspection.set(j2v.where("name") == "diagnosis", target=False)

Inspect the loaded model before serving. The plot should match the schema you expect to expose.

In [5]:
inspection.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  24,935                                │
│  Schema map                                                            arrays  2                                     │
│                                                                        fields  3                                     │
│                                                                       targets  0                                     │
│                                                                        embeds  1                                     │
│                               

For serving, the deployment mutates `diagnosis` with `target=False`. The field remains part of the model schema, but incoming requests do not need to include the answer you want the model to predict.

In [6]:
deployment = (
    Deployment(checkpoint=str(CHECKPOINT))
    .set(j2v.where("name") == "diagnosis", target=False)
    .forge(request=Request)
)

print("deployment configured")

deployment configured


The final cell is guarded by an environment variable so documentation builds configure the server object without starting a long-running process.

In [7]:
if os.environ.get("JSON2VEC_SERVE") == "1":
    deployment.serve()